# Lecture 13: Reinforcement Learning — Exercises

Companion to the note [Noter små](../Noter%20små.md).

Practise model-free reinforcement learning: the relation between $V^*$ and $Q^*$, Q-learning updates traced by hand (exam style), online averaging and learning rates, ε-greedy exploration, and then tabular Q-learning in NumPy on a two-state problem and on the adventurer grid world from Lecture 12, with evaluation against the optimal policy.

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

# --- Two-state environment (exercise-sheet style). States s1=0, s2=1; actions a1=0, a2=1.
#     a1 always leads to s1, a2 always leads to s2. The reward depends on the state you come from and the action:
R_two = np.array([[2.0, -1.0],    # R(s1,a1), R(s1,a2)
                  [6.0,  3.0]])   # R(s2,a1), R(s2,a2)
def two_state_step(s, a):
    return a, R_two[s, a]          # next state, reward

# --- The grid world of Lecture 12 (cells (col,row), gold (3,1) terminal +10, snake (3,2) -5, else -0.1,
#     move succeeds with prob. 0.8, otherwise slip one cell east; walls = stay).
cells = [(c, r) for r in (1, 2, 3) for c in (1, 2, 3)]
idx = {s: i for i, s in enumerate(cells)}
ACTIONS = ['N', 'E', 'S', 'W']
_DIR = {'N': (0, -1), 'E': (1, 0), 'S': (0, 1), 'W': (-1, 0)}
def _move(s, d):
    c, r = s[0] + d[0], s[1] + d[1]
    return (c, r) if 1 <= c <= 3 and 1 <= r <= 3 else s
P = np.zeros((4, 9, 9))
for _a, _n in enumerate(ACTIONS):
    for _s in cells:
        P[_a, idx[_s], idx[_move(_s, _DIR[_n])]] += 0.8
        P[_a, idx[_s], idx[_move(_s, (1, 0))]] += 0.2
terminal = np.zeros(9, bool); terminal[idx[(3, 1)]] = True
R = np.full(9, -0.1); R[idx[(3, 1)]] = 10.0; R[idx[(3, 2)]] = -5.0
gamma = 0.9
start = idx[(1, 3)]

def grid_step(s, a, rng):
    """Sample the environment: returns (next_state, reward, done). The agent does NOT get to see P or R.
    The reward is R(s) of the state being left; on reaching the terminal gold square the episode ends and
    its reward R(gold) is the value of the next state."""
    s2 = int(rng.choice(9, p=P[a, s]))
    return s2, R[s], bool(terminal[s2])

def show_policy(pi):
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    for r in range(3):
        print(' '.join('G' if terminal[3 * r + c] else arrows[int(pi[3 * r + c])] for c in range(3)))

## Part A · Concepts

### Exercise 1 · Why learn Q instead of V?
*🧠 Concept · ★☆☆*

1. What is the difference between the setting of Lecture 12 (MDPs) and reinforcement learning?
2. Write $Q^*(s,a)$ in terms of $V^*$ and $V^*$ in terms of $Q^*$ (as in the note).
3. An agent knows $V^*$ but not $P(s'\mid s,a)$. Can it act optimally? What if it knows $Q^*$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

To act greedily with respect to $V^*$ you need to compute $\sum_{s'}P(s'\mid s,a)V^*(s')$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. In an MDP the agent knows the transition probabilities and the reward function and can **plan** (value/policy iteration). In RL it does
   not know them and must **learn from experience** $\langle s, a, r, s'\rangle$ by interacting with the environment.
2. $Q^*(s,a) = R(s) + \gamma\sum_{s'}P(s'\mid s,a)V^*(s')$ and $V^*(s) = \max_aQ^*(s,a)$.
3. With $V^*$ only, choosing $\arg\max_a\sum_{s'}P(s'\mid s,a)V^*(s')$ requires the model, so no. With $Q^*$ it simply takes
   $\arg\max_aQ^*(s,a)$: no model needed. That is why model-free RL learns Q-values.

</details>

### Exercise 2 · Exploration versus exploitation
*🧠 Concept · ★☆☆*

1. Define exploration and exploitation and explain why an RL agent needs both.
2. Describe ε-greedy action selection. Why is ε often decreased over time?
3. Why is random exploration problematic when learning in the real world, and what is the usual way around it? What is its drawback?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of a self-driving car learning by trial and error at rush hour.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. *Exploration*: try actions whose consequences are not yet well known (needed to find the optimum and for Q-learning to converge).
   *Exploitation*: choose the action that currently looks best (needed to collect reward and to train efficiently).
2. With probability ε pick a uniformly random action, otherwise the greedy one $\arg\max_aQ(s,a)$. Early on the estimates are bad, so
   exploring a lot is useful; later the agent knows more and should mostly exploit, so ε is decayed.
3. Random actions in the real world can be dangerous (harm people or the robot) and experience is expensive. Use a **simulator**; but building a
   faithful simulator requires the model, and if you have the model you might as well use planning (Lecture 12).

</details>

### Exercise 3 · On-policy versus off-policy
*🧠 Concept · ★★★*

Q-learning uses the target $r + \gamma\max_{a'}Q(s',a')$. SARSA uses $r + \gamma Q(s',a')$ where $a'$ is the action the agent actually takes next
(e.g. ε-greedy).

1. Which of them learns the value of the greedy policy, and which the value of the policy that is actually followed?
2. In the grid world, an agent explores with ε = 0.3. Which algorithm would learn to keep a larger distance from the snake, and why?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Q-learning's target assumes optimal behaviour from $s'$ on, regardless of what the agent will really do.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Q-learning is **off-policy**: its target uses the max, so it learns $Q^*$ (the greedy policy's values) while behaving ε-greedily.
   SARSA is **on-policy**: it learns the values of the ε-greedy policy it actually follows.
2. SARSA. Its values include the cost of the random moves the agent will keep making, so cells next to the snake look worse and it prefers
   safer routes. Q-learning assumes perfect behaviour later and learns the shorter, riskier route, which is optimal only once exploration stops.

</details>

## Part B · By hand

### Exercise 4 · Tracing Q-learning updates
*✍️ By hand · ★★☆*

A 3 × 2 grid (columns 1–3, rows 1–2) with unknown dynamics. All Q-values start at 0, $\alpha = 0.5$, $\gamma = 0.9$. Starting in $(1,2)$ the agent experiences:

1. north, arrives in $(1,1)$, reward $-1$
2. east, arrives in $(2,1)$, reward $+4$
3. south, arrives in $(2,2)$, reward $-2$
4. west, arrives in $(1,2)$, reward $+3$
5. north, arrives in $(1,1)$, reward $-1$

Apply $Q(s,a) \leftarrow (1-\alpha)Q(s,a) + \alpha\big(r + \gamma\max_{a'}Q(s',a')\big)$ after each step. Give the updated value after each step, and
the action a greedy agent takes in $(1,2)$ afterwards.

In [ ]:
q_after = None   # list of the 5 updated Q-values, in order
greedy_12 = None # 'N', 'E', 'S' or 'W'

_Q = {}
_traj = [((1, 2), 'N', -1, (1, 1)), ((1, 1), 'E', 4, (2, 1)), ((2, 1), 'S', -2, (2, 2)), ((2, 2), 'W', 3, (1, 2)), ((1, 2), 'N', -1, (1, 1))]
_ref = []
for _s, _a, _r, _s2 in _traj:
    _best = max(_Q.get((_s2, b), 0.0) for b in ACTIONS)
    _Q[(_s, _a)] = 0.5 * _Q.get((_s, _a), 0.0) + 0.5 * (_r + 0.9 * _best); _ref.append(_Q[(_s, _a)])
check('updated values', q_after, _ref)
check('greedy action in (1,2)', greedy_12, max(ACTIONS, key=lambda b: _Q.get(((1, 2), b), 0.0)))

<details>
<summary><b>💡 Hint 1</b></summary>

Each update only changes $Q(s,a)$ for the state you **left** and the action you took.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

In step 5, $\max_{a'}Q((1,1),a')$ is no longer 0 because of step 2.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $Q((1,2),N) = 0.5\cdot0 + 0.5(-1 + 0.9\cdot0) = -0.5$
2. $Q((1,1),E) = 0.5(4 + 0) = 2$
3. $Q((2,1),S) = 0.5(-2 + 0) = -1$
4. $Q((2,2),W) = 0.5(3 + 0.9\cdot\max(-0.5,0,0,0)) = 1.5$
5. $Q((1,2),N) = 0.5\cdot(-0.5) + 0.5(-1 + 0.9\cdot2) = -0.25 + 0.4 = 0.15$

Greedy action in $(1,2)$: **N** (0.15 > 0). The reward of step 2 has propagated one step back.

```python
q_after = [-0.5, 2.0, -1.0, 1.5, 0.15]
greedy_12 = 'N'
```

</details>

### Exercise 5 · Online averages and learning rates
*✍️ By hand · ★☆☆*

1. Show that the update $\hat x_n = \hat x_{n-1} + \frac1n(x_n - \hat x_{n-1})$ computes the running mean. Use it on the stream $10, 20, 9, 5$.
2. Use instead a constant $\alpha = 0.5$, $\hat x_n = (1-\alpha)\hat x_{n-1} + \alpha x_n$, starting from $\hat x_0 = 0$. Give all four estimates.
3. Which weight does the constant-α estimate put on the first observation after $n$ steps, and why can that be desirable in RL?

In [ ]:
running_mean = None   # 4 estimates
const_alpha = None    # 4 estimates

_x = np.array([10, 20, 9, 5.])
check('running mean', running_mean, np.cumsum(_x) / np.arange(1, 5))
_e, _ref = 0.0, []
for _v in _x:
    _e = 0.5 * _e + 0.5 * _v; _ref.append(_e)
check('constant alpha', const_alpha, _ref)

<details>
<summary><b>💡 Hint</b></summary>

Induction: if $\hat x_{n-1}$ is the mean of $n-1$ numbers, then $\hat x_{n-1} + \frac1n(x_n - \hat x_{n-1}) = \frac{(n-1)\hat x_{n-1} + x_n}{n}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\frac{(n-1)\hat x_{n-1} + x_n}{n}$ is the mean of $n$ numbers. Stream: $10, 15, 13, 11$.
2. $5, 12.5, 10.75, 7.875$.
3. The first observation keeps weight $\alpha(1-\alpha)^{n-1}$, which decays exponentially: recent experience counts more. In RL the targets
   $r + \gamma\max Q(s',\cdot)$ change as $Q$ improves, so forgetting old, outdated targets is useful. Q-learning is an online average
   of these targets.

```python
running_mean = [10, 15, 13, 11]
const_alpha = [5, 12.5, 10.75, 7.875]
```

</details>

### Exercise 6 · ε-greedy probabilities
*✍️ By hand · ★☆☆*

In a state with 4 actions and a unique greedy action, an agent uses ε-greedy with $\epsilon = 0.2$, where the random choice is uniform over **all**
4 actions (including the greedy one).

1. With which probability does it pick the greedy action, and each other action?
2. Over 50 steps in that state, how many times do you expect it to try a specific non-greedy action?

In [ ]:
p_greedy = None
p_other = None
expected_tries = None

check('P(greedy)', p_greedy, 0.8 + 0.2 / 4)
check('P(each other)', p_other, 0.2 / 4)
check('expected tries', expected_tries, 50 * 0.2 / 4)

<details>
<summary><b>💡 Hint</b></summary>

The greedy action can be chosen in two ways: exploiting, or exploring and happening to pick it.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $P(\text{greedy}) = 0.8 + 0.2\cdot\tfrac14 = 0.85$; each other action $0.2\cdot\tfrac14 = 0.05$.
2. $50\cdot0.05 = 2.5$ times. Exploration is slow: rarely-visited state–action pairs need many visits before their Q-values are reliable.

```python
p_greedy = 0.85
p_other = 0.05
expected_tries = 2.5
```

</details>

### Exercise 7 · V* and Q* of the two-state problem
*✍️ By hand · ★★☆*

Two states $s_1, s_2$ and two actions: $a_1$ always leads to $s_1$, $a_2$ always to $s_2$. Rewards (depending on the state you come from):
$R(s_1,a_1) = 2$, $R(s_1,a_2) = -1$, $R(s_2,a_1) = 6$, $R(s_2,a_2) = 3$. $\gamma = 0.9$, no terminal states.

1. Write the transition function.
2. Compute the value of each of the four deterministic policies in $s_1$ and $s_2$ (solve the Bellman equations for each), and find $V^*$.
3. Compute $Q^*(s,a) = R(s,a) + \gamma V^*(s')$ for all four pairs and check $V^*(s) = \max_aQ^*(s,a)$.

In [ ]:
V_star_two = None   # [V*(s1), V*(s2)]
Q_star_two = None   # 2x2: rows = states, columns = actions

_V = np.zeros(2)
for _ in range(2000):
    _V = (R_two + 0.9 * _V[None, :]).max(1)
check('V*', V_star_two, _V, tol=1e-3)
check('Q*', Q_star_two, R_two + 0.9 * _V[None, :], tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$P(s_1\mid s,a_1) = 1$ and $P(s_2\mid s,a_2) = 1$ for both $s$. The policy is a pair (action in $s_1$, action in $s_2$).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Policy $(a_2, a_2)$: $V(s_2) = 3 + 0.9V(s_2) \Rightarrow V(s_2) = 30$ and $V(s_1) = -1 + 0.9\cdot30$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Deterministic: $P(s_1\mid s, a_1) = 1$, $P(s_2\mid s, a_2) = 1$ for $s\in\{s_1,s_2\}$.
2. * $(a_1, a_1)$: $V(s_1) = 2/0.1 = 20$; $V(s_2) = 6 + 0.9\cdot20 = 24$.
   * $(a_1, a_2)$: $V(s_1) = 20$, $V(s_2) = 30$.
   * $(a_2, a_1)$ (alternate): $V(s_1) = -1 + 0.9V(s_2)$, $V(s_2) = 6 + 0.9V(s_1)$ → $V(s_1) = 4.4/0.19 \approx 23.16$, $V(s_2) \approx 26.84$.
   * $(a_2, a_2)$: $V(s_2) = 30$, $V(s_1) = -1 + 27 = 26$.

   $V^* = (26, 30)$: move to $s_2$ (accepting $-1$ once) and stay there.
3. $Q^*(s_1,a_1) = 2 + 0.9\cdot26 = 25.4$, $Q^*(s_1,a_2) = -1 + 0.9\cdot30 = 26$, $Q^*(s_2,a_1) = 6 + 23.4 = 29.4$, $Q^*(s_2,a_2) = 3 + 27 = 30$.
   Row maxima are 26 and 30 = $V^*$ ✔. Note that the optimal first step in $s_1$ has the **worst** immediate reward.

```python
V_star_two = [26.0, 30.0]
Q_star_two = [[25.4, 26.0], [29.4, 30.0]]
```

</details>

### Exercise 8 · Learning-rate schedules
*✍️ By hand · ★★☆*

Q-learning converges to $Q^*$ if every state–action pair is visited infinitely often and the learning rates satisfy
$\sum_t\alpha_t = \infty$ and $\sum_t\alpha_t^2 < \infty$. Which schedules satisfy both conditions?

(a) $\alpha_t = 0.1$  (b) $\alpha_t = 1/t$  (c) $\alpha_t = 1/t^2$  (d) $\alpha_t = 1/\sqrt t$  (e) $\alpha_t = 1/t^{0.7}$

Store the letters of the valid ones in `valid_schedules`, and give the intuition behind both conditions.

In [ ]:
valid_schedules = None   # e.g. ['a', 'c']

_p = {'a': 0.0, 'b': 1.0, 'c': 2.0, 'd': 0.5, 'e': 0.7}   # alpha_t = t^(-p)
_ref = sorted(k for k, p in _p.items() if p <= 1 and 2 * p > 1)
check('valid schedules', None if valid_schedules is None else sorted(valid_schedules), _ref)

<details>
<summary><b>💡 Hint</b></summary>

$\sum_t t^{-p}$ diverges exactly when $p \le 1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Valid: **(b)** and **(e)**. (a) $\sum\alpha_t^2 = \infty$; (c) $\sum\alpha_t < \infty$; (d) $\sum 1/t = \infty$ for the squares.
For $t^{-p}$ we need $p \le 1$ (first condition) and $2p > 1$ (second), i.e. $\tfrac12 < p \le 1$.

Intuition: $\sum\alpha_t = \infty$ — never stop learning, so you can move arbitrarily far from a bad initial guess.
$\sum\alpha_t^2 < \infty$ — the steps eventually become small enough that the noise of individual samples averages out.

```python
valid_schedules = ['b', 'e']
```

</details>

## Part C · Code

### Exercise 9 · The Q-learning update
*💻 Code · ★☆☆*

Implement `q_update(Q, s, a, r, s2, alpha, gamma, done=False)` that updates the table `Q` (shape states × actions) **in place** and returns the
new value. When `done` is true there is no next state to bootstrap from, so the target is just `r`. Replay the trajectory of the by-hand
exercise (map cells of the 3×2 grid to indices with `cells32` below) and print the final table.

In [ ]:
cells32 = [(c, r) for r in (1, 2) for c in (1, 2, 3)]
i32 = {s: i for i, s in enumerate(cells32)}

def q_update(Q, s, a, r, s2, alpha, gamma, done=False):
    # TODO
    return None

Q32 = np.zeros((6, 4))
traj = [((1, 2), 'N', -1, (1, 1)), ((1, 1), 'E', 4, (2, 1)), ((2, 1), 'S', -2, (2, 2)),
        ((2, 2), 'W', 3, (1, 2)), ((1, 2), 'N', -1, (1, 1))]
for s, a, r, s2 in traj:
    q_update(Q32, i32[s], ACTIONS.index(a), r, i32[s2], 0.5, 0.9)

_done = bool(Q32.any())
check('Q((1,2),N)', Q32[i32[(1, 2)], 0] if _done else None, 0.15); check('Q((2,2),W)', Q32[i32[(2, 2)], 3] if _done else None, 1.5)
_Qt = np.zeros((1, 2));
check('terminal update ignores the next state', None if q_update(_Qt, 0, 0, 5.0, 0, 0.5, 0.9, done=True) is None else _Qt[0, 0], 2.5)

<details>
<summary><b>💡 Hint</b></summary>

`target = r if done else r + gamma * Q[s2].max()`; `Q[s, a] += alpha * (target - Q[s, a])` (the TD form).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$Q \leftarrow Q + \alpha(\text{target} - Q)$ is the same as $(1-\alpha)Q + \alpha\,\text{target}$: the slides call
$\text{target} - Q$ the **temporal difference** (TD error).

```python
cells32 = [(c, r) for r in (1, 2) for c in (1, 2, 3)]
i32 = {s: i for i, s in enumerate(cells32)}

def q_update(Q, s, a, r, s2, alpha, gamma, done=False):
    target = r if done else r + gamma * Q[s2].max()
    Q[s, a] += alpha * (target - Q[s, a])
    return Q[s, a]

Q32 = np.zeros((6, 4))
traj = [((1, 2), 'N', -1, (1, 1)), ((1, 1), 'E', 4, (2, 1)), ((2, 1), 'S', -2, (2, 2)),
        ((2, 2), 'W', 3, (1, 2)), ((1, 2), 'N', -1, (1, 1))]
for s, a, r, s2 in traj:
    q_update(Q32, i32[s], ACTIONS.index(a), r, i32[s2], 0.5, 0.9)
print(Q32)
```

</details>

### Exercise 10 · ε-greedy action selection
*💻 Code · ★☆☆*

Implement `eps_greedy(q_row, eps, rng)`: with probability `eps` return a uniformly random action index, otherwise the index of the largest
value (break ties at random). Sample 20 000 actions for `q_row = [1.0, 3.0, 2.0, 0.0]` with `eps = 0.2` and store the empirical frequencies in `freqs`.

In [ ]:
def eps_greedy(q_row, eps, rng):
    # TODO
    return None

rng_e = np.random.default_rng(0)
freqs = None

check('frequencies close to (0.05, 0.85, 0.05, 0.05)', freqs, [0.05, 0.85, 0.05, 0.05], tol=0.01)

<details>
<summary><b>💡 Hint</b></summary>

`best = np.flatnonzero(q_row == q_row.max())`, then `rng.choice(best)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The frequencies match the by-hand computation (0.85 for the greedy action). Random tie-breaking matters at the start, when all Q-values are 0;
always picking the first index would bias exploration (as the next exercise shows).

```python
def eps_greedy(q_row, eps, rng):
    q_row = np.asarray(q_row)
    if rng.random() < eps:
        return int(rng.integers(len(q_row)))
    return int(rng.choice(np.flatnonzero(q_row == q_row.max())))

rng_e = np.random.default_rng(0)
acts = [eps_greedy(np.array([1.0, 3.0, 2.0, 0.0]), 0.2, rng_e) for _ in range(20000)]
freqs = np.bincount(acts, minlength=4) / 20000
print(freqs)
```

</details>

### Exercise 11 · The greedy trap
*💻 Code · ★★☆*

Run Q-learning on the two-state problem (`two_state_step`) for 2000 steps from $s_1$ with $\alpha = 0.9$, $\gamma = 0.9$, **purely greedy** action
selection, ties broken in favour of $a_1$ (`np.argmax`). Store the final table in `Q_greedy`. Then repeat with ε-greedy, $\epsilon = 0.3$
(`np.random.default_rng(0)`, α = 0.1, 20 000 steps) and store it in `Q_eps`. Compare both with $Q^*$ from the by-hand exercise.

In [ ]:
Q_greedy = None
Q_eps = None

_V = np.zeros(2)
for _ in range(2000):
    _V = (R_two + 0.9 * _V[None, :]).max(1)
_Qs = R_two + 0.9 * _V[None, :]
check('greedy agent never tries a2 in s1', None if Q_greedy is None else Q_greedy[0, 1], 0.0)
check('greedy agent believes Q(s1,a1) = 20', None if Q_greedy is None else Q_greedy[0, 0], 20.0, tol=1e-3)
check('epsilon-greedy learns Q*', Q_eps, _Qs, tol=0.05)

<details>
<summary><b>💡 Hint</b></summary>

Reuse `q_update` and `eps_greedy`. The environment is deterministic, so a constant α still converges.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The greedy agent takes $a_1$ (tie at 0), gets $+2$, so $Q(s_1,a_1) > 0 = Q(s_1,a_2)$ and it **never** tries $a_2$, whose immediate reward $-1$ hides the
better long-term option. It converges to $Q(s_1,a_1) = 2/(1-0.9) = 20$, the value of a suboptimal policy, and believes it is optimal.
With ε-greedy exploration every pair keeps being tried and the table converges to $Q^* = \begin{pmatrix}25.4 & 26\\ 29.4 & 30\end{pmatrix}$.

```python
Q_greedy = np.zeros((2, 2)); s = 0
for _ in range(2000):
    a = int(np.argmax(Q_greedy[s]))
    s2, r = two_state_step(s, a)
    q_update(Q_greedy, s, a, r, s2, 0.9, 0.9); s = s2

Q_eps = np.zeros((2, 2)); s = 0; rng_q = np.random.default_rng(0)
for _ in range(20000):
    a = eps_greedy(Q_eps[s], 0.3, rng_q)
    s2, r = two_state_step(s, a)
    q_update(Q_eps, s, a, r, s2, 0.1, 0.9); s = s2
print(Q_greedy); print(Q_eps)
```

</details>

### Exercise 12 · Q-learning in the grid world
*💻 Code · ★★★*

Learn the adventurer's policy **without** using `P` or `R` directly, only through `grid_step`. Run 3000 episodes, each starting in a random
non-terminal cell and lasting at most 100 steps, with ε-greedy exploration (ε = 0.2) and the decaying learning rate
$\alpha = 1/N(s,a)^{0.6}$ where $N(s,a)$ counts visits. When `done`, the target is $r + \gamma R(\text{gold})$, i.e. call
`q_update(Q, s, a, r + gamma * 10.0, s2, alpha, gamma, done=True)`. Use `np.random.default_rng(0)`.
Store the table in `Q_grid` and print its greedy policy.

In [ ]:
Q_grid = None

_V = np.zeros(9)
for _ in range(2000):
    _V = np.where(terminal, R, R + gamma * (P @ _V).max(0))
_pi = (P @ _V).argmax(0); _nt = ~terminal
if Q_grid is None:
    check('greedy policy equals the optimal one', None, True)
else:
    check('greedy policy equals the optimal one', np.array_equal(Q_grid.argmax(1)[_nt], _pi[_nt]), True)
    check('max_a Q close to V*', Q_grid.max(1)[_nt], _V[_nt], tol=0.5)

<details>
<summary><b>💡 Hint 1</b></summary>

Episode loop: pick `a = eps_greedy(Q[s], 0.2, rng)`, call `grid_step(s, a, rng)`, increment `N[s, a]`, update, stop when `done`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Random non-terminal start: `rng.choice(np.flatnonzero(~terminal))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Q-learning recovers the policy of value iteration from Lecture 12 (north along column 1, then east), and $\max_aQ(s,a)$ is close to $V^*(s)$,
without ever looking at the transition probabilities: rare outcomes such as the 0.2 slip are simply sampled less often. The values of rarely
visited cells (e.g. the snake) are the least accurate. Random starts help to visit every state often enough.

```python
rng_g = np.random.default_rng(0)
Q_grid = np.zeros((9, 4)); N_grid = np.zeros((9, 4))
for ep in range(3000):
    s = int(rng_g.choice(np.flatnonzero(~terminal)))
    for t in range(100):
        a = eps_greedy(Q_grid[s], 0.2, rng_g)
        s2, r, done = grid_step(s, a, rng_g)
        N_grid[s, a] += 1
        alpha = 1 / N_grid[s, a] ** 0.6
        if done:
            q_update(Q_grid, s, a, r + gamma * 10.0, s2, alpha, gamma, done=True)
            break
        q_update(Q_grid, s, a, r, s2, alpha, gamma)
        s = s2
show_policy(Q_grid.argmax(1))
print(np.round(Q_grid.max(1).reshape(3, 3), 2))
```

</details>

### Exercise 13 · Evaluating the learned policy
*💻 Code · ★★☆*

Evaluate the greedy policy of `Q_grid` the way the lecture suggests (no exploration during evaluation): run 2000 episodes from the start cell
$(1,3)$ (at most 100 steps, `np.random.default_rng(1)`), compute each episode's discounted return
$\sum_t\gamma^tr_t$ (counting $\gamma^{T}\cdot10$ for reaching the gold at step $T$), and store the mean in `mc_return`.
Compare it with $V^*(1,3) \approx 5.83$.

In [ ]:
mc_return = None

_V = np.zeros(9)
for _ in range(2000):
    _V = np.where(terminal, R, R + gamma * (P @ _V).max(0))
check('Monte-Carlo return close to V*(start)', None if mc_return is None else abs(mc_return - _V[start]) < 0.3, True)

<details>
<summary><b>💡 Hint</b></summary>

Keep a running discount factor `disc *= gamma` after each step.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The average discounted return is within a few hundredths of $V^*(1,3) = 5.83$ (the standard error over 2000 episodes is about 0.03): the learned
policy is (near) optimal. Two evaluation views from the lecture: reward accumulated **while** learning (includes exploration cost), and quality
of the final greedy policy (this exercise).

```python
rng_m = np.random.default_rng(1)
pi_learned = Q_grid.argmax(1)
returns = []
for ep in range(2000):
    s, G, disc = start, 0.0, 1.0
    for t in range(100):
        s2, r, done = grid_step(s, int(pi_learned[s]), rng_m)
        G += disc * r; disc *= gamma
        if done:
            G += disc * 10.0
            break
        s = s2
    returns.append(G)
mc_return = float(np.mean(returns))
print(mc_return)
```

</details>

### Exercise 14 · Exploration in a multi-armed bandit
*💻 Code · ★★☆*

A one-state RL problem: 10 slot machines with true mean rewards `true_means` (below), each pull returns the mean plus $\mathcal N(0,1)$ noise.
Estimate each arm's value with the running mean ($\alpha = 1/n$) and select arms ε-greedily for 2000 pulls. For ε ∈ {0, 0.1}, store the fraction of
pulls of the best arm in `frac_best` (dict), averaged over 50 independent runs (`np.random.default_rng(run)`). Initial estimates are 0.

In [ ]:
true_means = np.random.default_rng(42).normal(0, 1, 10)
frac_best = None   # {0.0: ..., 0.1: ...}

if frac_best is not None:
    check('epsilon=0.1 finds the best arm more often than greedy', frac_best[0.1] > frac_best[0.0] + 0.1, True)
else:
    check('epsilon=0.1 finds the best arm more often than greedy', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Keep `counts` and `est` arrays of length 10. After pulling arm `a` with reward `x`: `counts[a] += 1; est[a] += (x - est[a]) / counts[a]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The greedy agent locks onto whichever arm first looks good and often never discovers the best one; ε = 0.1 keeps sampling all arms and pulls
the best arm far more often (about 34 % vs 60 % of the pulls here). Same lesson as the greedy trap, without states.

```python
true_means = np.random.default_rng(42).normal(0, 1, 10)
best_arm = int(np.argmax(true_means))
frac_best = {}
for eps in (0.0, 0.1):
    fr = []
    for run in range(50):
        r_b = np.random.default_rng(run)
        est, counts, n_best = np.zeros(10), np.zeros(10), 0
        for t in range(2000):
            a = eps_greedy(est, eps, r_b)
            x = true_means[a] + r_b.normal()
            counts[a] += 1; est[a] += (x - est[a]) / counts[a]
            n_best += a == best_arm
        fr.append(n_best / 2000)
    frac_best[eps] = float(np.mean(fr))
print(frac_best)
```

</details>

### Exercise 15 · Why tables do not scale
*✍️ By hand · ★★☆*

Tabular Q-learning stores $|S|\cdot|A|$ numbers.

1. How many entries does the Q-table of the 3×3 grid world have?
2. A Tetris board has 10 × 20 cells, each empty or filled, plus one of 7 falling pieces; ignore the piece position. Estimate $\log_{10}$ of the
   number of states, and of the table size with 9 actions (3 moves × 3 rotations, as on the exercise sheet).
3. How does deep Q-learning address this?

In [ ]:
n_entries_grid = None
log10_states_tetris = None
log10_table_tetris = None

check('grid Q-table', n_entries_grid, 9 * 4)
check('log10 #states', log10_states_tetris, 200 * np.log10(2) + np.log10(7), tol=0.05)
check('log10 table size', log10_table_tetris, 200 * np.log10(2) + np.log10(7) + np.log10(9), tol=0.05)

<details>
<summary><b>💡 Hint</b></summary>

$\log_{10}(2^{200}\cdot7) = 200\log_{10}2 + \log_{10}7$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $9\cdot4 = 36$ entries.
2. $2^{200}\cdot7$ states: $\log_{10} \approx 60.2 + 0.85 = 61.05$; with 9 actions $\approx 62.0$. Far beyond any memory, and almost every state
   would never be visited, so nothing could be learned for it.
3. Deep Q-learning replaces the table by a neural network $Q_\theta(s,a)$ (Lecture 06) that takes a state representation as input and
   **generalises** between similar states. It is trained towards the same targets $r + \gamma\max_{a'}Q_\theta(s',a')$.

```python
n_entries_grid = 36
log10_states_tetris = 200 * np.log10(2) + np.log10(7)
log10_table_tetris = log10_states_tetris + np.log10(9)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Noter små](../Noter%20små.md).*